# PhoBERT + SentiWordNet Baseline

## Mô tả
Hybrid model với **PhoBERT fine-tuned baseline** + 8 SentiWordNet features.

**PhoBERT Source:** `results/PhoBERT/baseline/models/phobert_model.pt` (đã fine-tune)

**Features:** pos_sum, neg_sum, pos_max, neg_max, pos_mean, neg_mean, coverage, polarity

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install transformers torch

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import numpy as np
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
set_seed(42)

văn bản in nghiêng## 2. Config

In [ ]:
class Config:
    BASE_DIR = '/content/drive/MyDrive/Student-Feedback-Sentiment-Analysis'
    DATA_DIR = f'{BASE_DIR}/data/processed'
    MODEL_TYPE = 'PhoBERT_Sentiwordnet'
    EXPERIMENT_TYPE = 'baseline'
    RESULTS_DIR = f'{BASE_DIR}/results/{MODEL_TYPE}/{EXPERIMENT_TYPE}'
    VISUALIZATIONS_DIR = f'{RESULTS_DIR}/visualizations'
    MODELS_DIR = f'{RESULTS_DIR}/models'
    SUMMARIES_DIR = f'{RESULTS_DIR}/summaries'
    ARTIFACTS_DIR = f'{RESULTS_DIR}/artifacts'
    MODEL_NAME = 'vinai/phobert-base'
    # Path to fine-tuned PhoBERT baseline model
    PHOBERT_MODEL_DIR = f'{BASE_DIR}/results/PhoBERT/baseline/models/phobert_model.pt'
    NUM_CLASSES = 3
    LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    SENTIWORDNET_FILE = f'{BASE_DIR}/data/sentiwordnet-dataset/VietSentiWordnet_Ver1.3.5.txt'

config = Config()
print(f'Results: {config.RESULTS_DIR}')
print(f'Visualizations: {config.VISUALIZATIONS_DIR}')
print(f'Models: {config.MODELS_DIR}')
print(f'Summaries: {config.SUMMARIES_DIR}')
print(f'Artifacts: {config.ARTIFACTS_DIR}')
print(f'PhoBERT Baseline: {config.PHOBERT_MODEL_DIR}')

# ── Create directories ───────────────────────────────────────────────────────
import os
for dir_path in [config.RESULTS_DIR, config.VISUALIZATIONS_DIR,
                 config.MODELS_DIR, config.SUMMARIES_DIR, config.ARTIFACTS_DIR]:
    os.makedirs(dir_path, exist_ok=True)
print('\n✓ All directories created successfully')

## 3. Load Data

In [ ]:
# Import from centralized data_utils module
import sys
sys.path.append(config.BASE_DIR)
from src.data_utils import load_data

train_texts, train_labels = load_data(config.DATA_DIR, 'train')
val_texts, val_labels = load_data(config.DATA_DIR, 'validation')
test_texts, test_labels = load_data(config.DATA_DIR, 'test')

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
import os

COLORS = ['#e74c3c', '#3498db', '#2ecc71']
plt.rcParams.update({'font.size': 11, 'figure.facecolor': 'white'})

# ── Label Distribution ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, labels) in zip(axes, [('Train', train_labels),
                                       ('Validation', val_labels),
                                       ('Test', test_labels)]):
    counts = Counter(labels)
    cls_names  = [config.LABEL_MAP[i] for i in range(config.NUM_CLASSES)]
    cls_counts = [counts.get(i, 0) for i in range(config.NUM_CLASSES)]
    bars = ax.bar(cls_names, cls_counts, color=COLORS, edgecolor='white')
    for bar, cnt in zip(bars, cls_counts):
        pct = cnt / len(labels) * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(cls_counts)*0.02,
                f'{cnt}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
    ax.set_title(f'{name}  (n={len(labels)})', fontweight='bold')
    ax.set_xlabel('Sentiment')
    ax.set_ylabel('Count')
plt.suptitle('Label Distribution — PhoBERT + SentiWordNet', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.VISUALIZATIONS_DIR, 'label_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: label_distribution.png')

# ── Text Length Distribution ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, texts) in zip(axes, [('Train', train_texts),
                                      ('Validation', val_texts),
                                      ('Test', test_texts)]):
    lengths = [len(t.split()) for t in texts]
    ax.hist(lengths, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(lengths), color='red', linestyle='--',
               label=f'Mean: {np.mean(lengths):.1f}')
    ax.axvline(np.median(lengths), color='orange', linestyle='--',
               label=f'Median: {np.median(lengths):.0f}')
    ax.set_title(f'{name} Set', fontweight='bold')
    ax.set_xlabel('Word Count')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)
plt.suptitle('Text Length Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.VISUALIZATIONS_DIR, 'text_length_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: text_length_distribution.png')

# ── Class Imbalance ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
split_dicts  = {'Train': Counter(train_labels),
                'Val':   Counter(val_labels),
                'Test':  Counter(test_labels)}
x_labels = list(split_dicts.keys())
totals   = [sum(v.values()) for v in split_dicts.values()]
bottoms  = np.zeros(len(x_labels))
for cls_idx, cls_name in config.LABEL_MAP.items():
    pcts = [split_dicts[s].get(cls_idx, 0) / t * 100
            for s, t in zip(x_labels, totals)]
    axes[0].bar(x_labels, pcts, bottom=bottoms, label=cls_name,
                color=COLORS[cls_idx])
    for i, (p, b) in enumerate(zip(pcts, bottoms)):
        if p > 3:
            axes[0].text(i, b + p/2, f'{p:.1f}%', ha='center', va='center',
                         color='white', fontweight='bold', fontsize=10)
    bottoms += np.array(pcts)
axes[0].set_ylabel('Percentage (%)')
axes[0].set_title('Class Distribution per Split (%)', fontweight='bold')
axes[0].legend(loc='upper right')

train_vals = [Counter(train_labels).get(i, 0) for i in range(config.NUM_CLASSES)]
axes[1].pie(train_vals,
            labels=[config.LABEL_MAP[i] for i in range(config.NUM_CLASSES)],
            autopct='%1.1f%%', colors=COLORS, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Train Set Class Distribution', fontweight='bold')
plt.suptitle('Class Imbalance Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.VISUALIZATIONS_DIR, 'class_imbalance.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: class_imbalance.png')


## 4. Load SentiWordNet

In [ ]:
# Import from centralized data_utils module
from src.data_utils import load_sentiwordnet

word_to_scores = load_sentiwordnet(config.SENTIWORDNET_FILE)
print(f'Loaded {len(word_to_scores)} words')

In [ ]:
# Import from centralized data_utils module
from src.data_utils import get_swn_features, extract_swn_features_batch, SWN_FEATURE_NAMES

print('Extracting SentiWordNet features for train set...')
train_swn = extract_swn_features_batch(train_texts, word_to_scores)
print(f'Feature matrix shape: {train_swn.shape}')

# ── Feature Distribution per Class ──────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()
for feat_idx, (ax, feat_name) in enumerate(zip(axes, SWN_FEATURE_NAMES)):
    for cls_idx in range(config.NUM_CLASSES):
        mask = np.array(train_labels) == cls_idx
        ax.hist(train_swn[mask, feat_idx], bins=30, alpha=0.6,
                label=config.LABEL_MAP[cls_idx],
                color=COLORS[cls_idx], edgecolor='none')
    ax.set_title(feat_name, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
plt.suptitle('SentiWordNet Feature Distribution by Class (Train)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.VISUALIZATIONS_DIR, 'swn_feature_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: swn_feature_distribution.png')

# ── Mean Feature Values per Class ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(SWN_FEATURE_NAMES))
width = 0.25
for cls_idx in range(config.NUM_CLASSES):
    mask  = np.array(train_labels) == cls_idx
    means = train_swn[mask].mean(axis=0)
    ax.bar(x + cls_idx * width, means, width,
           label=config.LABEL_MAP[cls_idx],
           color=COLORS[cls_idx], edgecolor='white')
ax.set_xticks(x + width)
ax.set_xticklabels(SWN_FEATURE_NAMES, rotation=20, ha='right')
ax.set_title('Mean SentiWordNet Feature Values per Class (Train)', fontweight='bold')
ax.set_ylabel('Mean Value')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(config.VISUALIZATIONS_DIR, 'swn_feature_means.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: swn_feature_means.png')
print(f'\n✓ All visualizations saved to:\n  {config.VISUALIZATIONS_DIR}')

## 5. PhoBERT Feature Extraction (Fine-tuned Baseline)

In [ ]:
# Define PhoBERT Classifier class (same architecture as baseline)
class PhoBERTClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.1):
        super(PhoBERTClassifier, self).__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits, pooled_output

    def extract_embeddings(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
            pooled_output = outputs.last_hidden_state[:, 0, :]
        return pooled_output

# Function to safely load model checkpoint
def load_model_safe(model, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(checkpoint, dict):
        if 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
        elif 'model_state_dict' in checkpoint:
            state_dict = checkpoint['model_state_dict']
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint
    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('module.'):
            new_state_dict[k[7:]] = v
        else:
            new_state_dict[k] = v
    model.load_state_dict(new_state_dict)
    return model

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
print(f'Tokenizer loaded: {config.MODEL_NAME}')

# Load fine-tuned PhoBERT baseline model
print(f'📥 Loading PhoBERT Baseline model from: {config.PHOBERT_MODEL_DIR}')
phobert_model = PhoBERTClassifier(model_name=config.MODEL_NAME, num_classes=config.NUM_CLASSES)
phobert_model = load_model_safe(phobert_model, config.PHOBERT_MODEL_DIR, config.DEVICE)
phobert_model = phobert_model.to(config.DEVICE)

# Freeze all parameters - only use for feature extraction
for param in phobert_model.parameters():
    param.requires_grad = False
phobert_model.eval()
print(f'✅ PhoBERT model loaded and frozen on {config.DEVICE}')

def extract_phobert_embeddings(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(config.DEVICE)
        with torch.no_grad():
            embeddings_batch = phobert_model.extract_embeddings(inputs['input_ids'], inputs['attention_mask'])
            embeddings.append(embeddings_batch.cpu().numpy())
        if (i // batch_size + 1) % 10 == 0:
            print(f"  Processed {i + len(batch_texts)}/{len(texts)} samples")
    return np.vstack(embeddings)

print("Extracting PhoBERT embeddings for train set...")
train_phobert = extract_phobert_embeddings(train_texts)
print(f"Train embeddings shape: {train_phobert.shape}")
print("Extracting PhoBERT embeddings for validation set...")
val_phobert = extract_phobert_embeddings(val_texts)
print(f"Val embeddings shape: {val_phobert.shape}")
print("Extracting PhoBERT embeddings for test set...")
test_phobert = extract_phobert_embeddings(test_texts)
print(f"Test embeddings shape: {test_phobert.shape}")

## 6. Combine Features

In [ ]:
# Import batch extraction from centralized module
from src.data_utils import extract_swn_features_batch

print("Extracting SentiWordNet features for validation set...")
val_swn = extract_swn_features_batch(val_texts, word_to_scores)
print(f"Val SentiWordNet features shape: {val_swn.shape}")

print("Extracting SentiWordNet features for test set...")
test_swn = extract_swn_features_batch(test_texts, word_to_scores)
print(f"Test SentiWordNet features shape: {test_swn.shape}")

train_features = np.hstack([train_phobert, train_swn])
val_features = np.hstack([val_phobert, val_swn])
test_features = np.hstack([test_phobert, test_swn])

print(f"\nCombined feature shapes:")
print(f"  Train: {train_features.shape}")
print(f"  Val:   {val_features.shape}")
print(f"  Test:  {test_features.shape}")

## 7. Train Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import time, joblib

print("Training LogisticRegression model...")
start_time = time.time()
model = LogisticRegression(max_iter=1000, C=1.0, random_state=42, n_jobs=-1)
model.fit(train_features, train_labels)
train_time = time.time() - start_time
print(f"Training completed in {train_time:.1f} seconds")
model_path = f"{config.MODELS_DIR}/phobert_swn_baseline_model.pkl"
joblib.dump(model, model_path)
print(f"Model saved to: {model_path}")

## 8. Evaluation

In [ ]:
def evaluate_model(model, features, labels, split_name):
    y_pred = model.predict(features)
    accuracy = accuracy_score(labels, y_pred)
    f1_macro = f1_score(labels, y_pred, average="macro")
    precision_macro = precision_score(labels, y_pred, average="macro")
    recall_macro = recall_score(labels, y_pred, average="macro")
    f1_per_class = f1_score(labels, y_pred, average=None)
    precision_per_class = precision_score(labels, y_pred, average=None)
    recall_per_class = recall_score(labels, y_pred, average=None)
    print(f"\n{"="*50}")
    print(f"{split_name} Results:")
    print(f"{"="*50}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"F1 (macro): {f1_macro:.4f}")
    print(f"Precision (macro): {precision_macro:.4f}")
    print(f"Recall (macro): {recall_macro:.4f}")
    print(f"\nPer-Class Metrics:")
    for i, name in config.LABEL_MAP.items():
        print(f"  {name}: P={precision_per_class[i]:.4f}, R={recall_per_class[i]:.4f}, F1={f1_per_class[i]:.4f}")
    cm = confusion_matrix(labels, y_pred)
    print(f"\nConfusion Matrix:")
    print(cm)
    return {"accuracy": accuracy, "f1_macro": f1_macro, "precision_macro": precision_macro,
            "recall_macro": recall_macro, "f1_per_class": f1_per_class.tolist(),
            "precision_per_class": precision_per_class.tolist(), "recall_per_class": recall_per_class.tolist(),
            "confusion_matrix": cm.tolist()}

results = {}
results["train"] = evaluate_model(model, train_features, train_labels, "Train")
results["val"] = evaluate_model(model, val_features, val_labels, "Validation")
results["test"] = evaluate_model(model, test_features, test_labels, "Test")

## 9. Visualizations

In [ ]:
import seaborn as sns
y_test_pred = model.predict(test_features)
cm = confusion_matrix(test_labels, y_test_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=config.LABEL_MAP.values(), yticklabels=config.LABEL_MAP.values(), ax=axes[0])
axes[0].set_title("Confusion Matrix (Count)", fontweight="bold")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues", xticklabels=config.LABEL_MAP.values(), yticklabels=config.LABEL_MAP.values(), ax=axes[1])
axes[1].set_title("Confusion Matrix (Normalized)", fontweight="bold")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
plt.suptitle("PhoBERT + SentiWordNet Baseline - Test Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{config.VISUALIZATIONS_DIR}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrix.png")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(config.NUM_CLASSES)
width = 0.25
test_results = results["test"]
precision = test_results["precision_per_class"]
recall = test_results["recall_per_class"]
f1 = test_results["f1_per_class"]
ax.bar(x - width, precision, width, label="Precision", color="#3498db")
ax.bar(x, recall, width, label="Recall", color="#2ecc71")
ax.bar(x + width, f1, width, label="F1", color="#e74c3c")
ax.set_xticks(x)
ax.set_xticklabels(config.LABEL_MAP.values())
ax.set_ylabel("Score")
ax.set_title("Per-Class Metrics (Test Set)", fontweight="bold")
ax.legend()
ax.set_ylim([0, 1.1])
for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
    ax.text(i - width, p + 0.02, f"{p:.3f}", ha="center", fontsize=9)
    ax.text(i, r + 0.02, f"{r:.3f}", ha="center", fontsize=9)
    ax.text(i + width, f + 0.02, f"{f:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{config.VISUALIZATIONS_DIR}/per_class_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: per_class_metrics.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
splits = ["Train", "Validation", "Test"]
accuracies = [results[s]["accuracy"] for s in ["train", "val", "test"]]
f1_scores = [results[s]["f1_macro"] for s in ["train", "val", "test"]]
x = np.arange(len(splits))
width = 0.35
ax.bar(x - width/2, accuracies, width, label="Accuracy", color="#3498db")
ax.bar(x + width/2, f1_scores, width, label="F1 (macro)", color="#2ecc71")
ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.set_ylabel("Score")
ax.set_title("Model Performance Across Splits", fontweight="bold")
ax.legend()
ax.set_ylim([0, 1.1])
for i, (acc, f1) in enumerate(zip(accuracies, f1_scores)):
    ax.text(i - width/2, acc + 0.02, f"{acc:.3f}", ha="center", fontsize=10)
    ax.text(i + width/2, f1 + 0.02, f"{f1:.3f}", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig(f"{config.VISUALIZATIONS_DIR}/performance_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: performance_comparison.png")

## 10. Save Summary

In [ ]:
import pandas as pd
from datetime import datetime
summary_data = []
for split in ["train", "val", "test"]:
    r = results[split]
    summary_data.append({"Split": split.capitalize(), "Accuracy": r["accuracy"], "F1_Macro": r["f1_macro"],
        "Precision_Macro": r["precision_macro"], "Recall_Macro": r["recall_macro"],
        "F1_Negative": r["f1_per_class"][0], "F1_Neutral": r["f1_per_class"][1], "F1_Positive": r["f1_per_class"][2]})
df_summary = pd.DataFrame(summary_data)
summary_path = f"{config.SUMMARIES_DIR}/summary.csv"
df_summary.to_csv(summary_path, index=False)
print(f"Summary saved to: {summary_path}")
print("\nResults Summary:")
print(df_summary.to_string(index=False))

In [ ]:
from datetime import datetime
training_log_path = f"{config.SUMMARIES_DIR}/training_results.txt"
with open(training_log_path, "w", encoding="utf-8") as f:
    f.write("="*50 + "\n")
    f.write("TRAINING RESULTS - PhoBERT + SentiWordNet Baseline\n")
    f.write("="*50 + "\n")
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("Model Type: PhoBERT_Sentiwordnet\n")
    f.write("Experiment: baseline\n")
    f.write("\n" + "-"*50 + "\n")
    f.write("HYPERPARAMETERS\n")
    f.write("-"*50 + "\n")
    f.write("Model: LogisticRegression\n")
    f.write("C (regularization): 1.0\n")
    f.write("Max Iterations: 1000\n")
    f.write("Random State: 42\n")
    f.write(f"PhoBERT Model: Fine-tuned baseline from {config.PHOBERT_MODEL_DIR}\n")
    f.write("PhoBERT Frozen: Yes\n")
    f.write(f"Feature Dimensions: PhoBERT={train_phobert.shape[1]}, SentiWordNet={train_swn.shape[1]}, Total={train_features.shape[1]}\n")
    f.write(f"SentiWordNet Words: {len(word_to_scores)}\n")
    f.write("\n" + "-"*50 + "\n")
    f.write("TRAINING RESULTS\n")
    f.write("-"*50 + "\n")
    r_train = results["train"]
    f.write(f"Train Accuracy: {r_train['accuracy']:.4f}\n")
    f.write(f"Train F1 (macro): {r_train['f1_macro']:.4f}\n")
    f.write("\n" + "-"*50 + "\n")
    f.write("VALIDATION RESULTS\n")
    f.write("-"*50 + "\n")
    r_val = results["val"]
    f.write(f"Val Accuracy: {r_val['accuracy']:.4f}\n")
    f.write(f"Val F1 (macro): {r_val['f1_macro']:.4f}\n")
    f.write("\n" + "-"*50 + "\n")
    f.write("TEST RESULTS\n")
    f.write("-"*50 + "\n")
    r_test = results["test"]
    f.write(f"Test Accuracy: {r_test['accuracy']:.4f}\n")
    f.write(f"Test F1 (macro): {r_test['f1_macro']:.4f}\n")
    f.write(f"Test Precision (macro): {r_test['precision_macro']:.4f}\n")
    f.write(f"Test Recall (macro): {r_test['recall_macro']:.4f}\n")
    f.write("\nPer-Class Metrics:\n")
    for i, name in config.LABEL_MAP.items():
        f.write(f"  {name}: Precision={r_test['precision_per_class'][i]:.4f}, Recall={r_test['recall_per_class'][i]:.4f}, F1={r_test['f1_per_class'][i]:.4f}\n")
    f.write("\n" + "-"*50 + "\n")
    f.write("CONFUSION MATRIX (Test)\n")
    f.write("-"*50 + "\n")
    cm = r_test["confusion_matrix"]
    for row in cm:
        f.write(f"[{row[0]:4d} {row[1]:4d} {row[2]:4d}]\n")
    f.write("\n" + "-"*50 + "\n")
    f.write("TRAINING TIME\n")
    f.write("-"*50 + "\n")
    f.write(f"Total Time: {train_time:.1f} seconds ({train_time/60:.1f} minutes)\n")
    f.write(f"Training Samples: {len(train_labels)}\n")
print(f"Training log saved to: {training_log_path}")

## 11. Final Summary

In [ ]:
print("\n" + "="*60)
print("PHOBERT + SENTIWORDNET BASELINE - COMPLETE SUMMARY")
print("="*60)
print(f"\nModel Architecture:")
print(f"  - PhoBERT (fine-tuned baseline): {config.PHOBERT_MODEL_DIR}")
print(f"  - SentiWordNet features: {train_swn.shape[1]}")
print(f"  - Classifier: LogisticRegression(C=1.0)")
print(f"\nFeature Dimensions:")
print(f"  - PhoBERT embeddings: {train_phobert.shape[1]}")
print(f"  - SentiWordNet features: {train_swn.shape[1]}")
print(f"  - Total: {train_features.shape[1]}")
print(f"\nTest Results:")
r_test = results["test"]
print(f"  - Accuracy: {r_test['accuracy']:.4f}")
print(f"  - F1 (macro): {r_test['f1_macro']:.4f}")
print(f"\nPer-Class F1 Scores:")
for i, name in config.LABEL_MAP.items():
    print(f"  - {name}: {r_test['f1_per_class'][i]:.4f}")
print(f"\nFiles saved:")
print(f"  - Model: {config.MODELS_DIR}/phobert_swn_baseline_model.pkl")
print(f"  - Summary CSV: {config.SUMMARIES_DIR}/summary.csv")
print(f"  - Training Log: {config.SUMMARIES_DIR}/training_results.txt")
print(f"  - Visualizations: {config.VISUALIZATIONS_DIR}/")
print("\n" + "="*60)